### Reassesing Quadratic Penalties

Here I am trying to build off of the work done by Bian, Litvinov, & Koumoustakos (2020) in which they apply quadratic penalties to enforce the area and volume constraints the meshes are subject to. 

These penalties look like

$E_A = \mu_A\frac{(A-A_0)^2}{A_0}$,

$E_V = \mu_V\frac{(V-V_0)^2}{V_0}$,

$E_T = \mu_T\sum\frac{(T-T_0)^2}{T_0}$,

for the constraints on membrane area, volume, and individual mesh triangle areas, which is added to maintain mesh quality over the course of optimization.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

import meshplot

import jax
import jax.numpy as jnp
import jaxopt
import optimistix

In [2]:
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_debug_nans", True)
jax.config.update('jax_log_compiles', False)

In [3]:
from triangulax import trigonometry as trg
from triangulax import mesh as msh
from triangulax.triangular import TriMesh
from triangulax.mesh import HeMesh, GeomMesh
from triangulax import geometry as geom
from triangulax import linops as lin
from triangulax import adjacency as adj

In [4]:
# define an energy to minimize with some quadratic penalties enforing a desired volume, area, and uniform triangle area

@jax.jit
def get_mean_curvature(ver, hemesh):

    """Function to compute mean curvature"""
    l_vec = lin.compute_cotan_laplace(ver, hemesh, ver)
    vor_A = geom.get_voronoi_areas(ver, hemesh)
    ver_norm_vec = geom.get_vertex_normals(ver, hemesh)
    h = -jnp.linalg.vecdot(l_vec,ver_norm_vec) / (2* vor_A)      # sphere has positive curvature
    return h

# Helfrich bending energy
@jax.jit
def helfrich_energy(ver, hemesh, h_0, kappa=1e-2):

    """Function to compute Helfrich with additional energy from overall area, bound volume, and triangle area"""
    h = get_mean_curvature(ver, hemesh)
    vor_A = geom.get_voronoi_areas(ver, hemesh)
    return jnp.sum((1/2)* kappa * (vor_A*(2*h - h_0)**2))

@jax.jit
def energy_function(ver,hemesh,h_0,A_0,V_0,T_0,kappa=1e-2,mu_A=2e0,mu_V=1e0,mu_T=1e0):

    """Function to compute Helfrich with additional energy from overall area, bound volume, and triangle area"""
    vor_A = geom.get_voronoi_areas(ver, hemesh)
    # compute Helfrich energy
    E_h = helfrich_energy(ver, hemesh, h_0, kappa)

    # compute area violation energy
    T = geom.get_triangle_areas(ver, hemesh)
    A = jnp.sum(vor_A)
    E_A = (mu_A/A_0)*(A-A_0)**2

    # compute volume violation energy
    V = geom.get_volume(ver, hemesh) #jnp.sum(trg.get_tetrahedron_volume(*ver[hemesh.faces.T]))/6
    E_V = (mu_V/V_0)*(V-V_0)**2

    # compute triangle violation energy - includes an extra term for Voronoi areas
    E_T = (mu_T/(2*T_0))*jnp.sum((vor_A-2*T_0)**2) + (mu_T/T_0)*jnp.sum((T-T_0)**2)

    # jax.debug.print("E_h: {E_h}, E_V: {E_V}, E_A: {E_A}, E_T: {E_T}",  E_h=E_h, E_A=E_A,E_V=E_V,E_T=E_T)

    return E_h + E_A + E_V + E_T

In [5]:
# Set up parameters (optimal volume and area) for optimization

R_0 = 1
S =  4*jnp.pi*R_0**2
V = 4*jnp.pi*R_0**3/3
A_0 = S
F = .7
V_0 = F*V
h_0 = 0

#### Load in a mesh

In this case I am taking a spherical (icosphere) mesh and deforming it to be a prolate or oblate ellipsoid with matching desired volume based upon desired final shape

In [12]:
# load in a mesh
mesh = TriMesh.read_obj("../test_meshes/sphere_fine.obj",dim =3) # sphere
hemesh = HeMesh.from_triangles(mesh.vertices.shape[0], mesh.faces)

  o Icosphere


In [15]:
geommesh = GeomMesh(*hemesh.n_items, vertices=mesh.vertices)

# modify vertex positions to give better initial conditions
oblate = False          # whether to squish (oblate) or stretch (prolate) the sphere to initialize at target volume
sf = 3*V_0/(4*jnp.pi)
s = 1.1                 # additional stretch for prolate shape

if oblate:
    geommesh.vertices = geommesh.vertices.at[:,1].multiply(sf)
else:
    geommesh.vertices = geommesh.vertices.at[:,[0,2]].multiply(jnp.sqrt(sf/s))
    geommesh.vertices = geommesh.vertices.at[:,1].multiply(s)

key = jax.random.PRNGKey(42)
noise = jax.random.normal(key, shape=geommesh.vertices.shape) *1e-2
geommesh.vertices = geommesh.vertices + noise

# set target triangle area as desired surface area/number of triangles in mesh
T_0 = A_0/len(mesh.faces)

In [ ]:
sf

In [16]:
meshplot.plot(geommesh.vertices, hemesh.faces, shading={"wireframe": True})

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0042690…

#### Run Simulation

In this scheme there is often some fudging to do regarding stopping tolerance/max iterations, etc to obtain an optimal shape

In [17]:
# optimization parameters
tol = 1e-2
max_iter = 5e3

# run an optimization
solver = jaxopt.LBFGS(fun=energy_function,tol=tol,maxiter=max_iter,verbose=True)#,max_stepsize=1e2)
res = solver.run(geommesh.vertices, hemesh = hemesh, h_0 = h_0, A_0 = A_0, V_0 = V_0, T_0 = T_0,
                 kappa=1e-2,mu_A=2e0,mu_V=2e0,mu_T=1e0)

INFO: jaxopt.LBFGS: Iter: 1 Gradient Norm (stopping criterion): 1.8320214820877314 Objective Value:2.3079288407704377  Stepsize:0.14338858862972576  Number Linesearch Iterations:2 
INFO: jaxopt.LBFGS: Iter: 2 Gradient Norm (stopping criterion): 1.6495846425000964 Objective Value:2.069192324486005  Stepsize:0.43016576588917727  Number Linesearch Iterations:2 
INFO: jaxopt.LBFGS: Iter: 3 Gradient Norm (stopping criterion): 1.5133821408038641 Objective Value:1.2952486803720418  Stepsize:0.6452486488337659  Number Linesearch Iterations:1 
INFO: jaxopt.LBFGS: Iter: 4 Gradient Norm (stopping criterion): 1.5367111239702715 Objective Value:1.0848285101387602  Stepsize:0.40456969181151736  Number Linesearch Iterations:2 
INFO: jaxopt.LBFGS: Iter: 5 Gradient Norm (stopping criterion): 0.9382624209647674 Objective Value:0.9677732228818849  Stepsize:0.606854537717276  Number Linesearch Iterations:1 
INFO: jaxopt.LBFGS: Iter: 6 Gradient Norm (stopping criterion): 0.4094553925734584 Objective Value:

#### Plot the initial and Final Mesh, Checking enforcing of volume/area constraints, final *Helfrich* Energy

In [ ]:
# calculating triangle areas, volume, total area|
T = geom.get_triangle_areas(res.params,hemesh)
vor_A = geom.get_voronoi_areas(res.params,hemesh)
A_f = jnp.sum(T)
V_f = geom.get_volume(res.params, hemesh) #jnp.sum(trg.get_tetrahedron_volume(*res.params[hemesh.faces.T]))/6

T_i = geom.get_triangle_areas(mesh.vertices,hemesh)
A_i = jnp.sum(T_i)
V_i = geom.get_volume(mesh.vertices, hemesh) #jnp.sum(trg.get_tetrahedron_volume(*mesh.vertices[hemesh.faces.T]))/6

# print volumes 
print("Volume")
print(f"Initial Volume: {V_i:.3e}")
print(f"Final Volume: {V_f:.3e}")
print(f"Target Volume: {V_0:.3e}\n")

# print area 
print("Area")
print(f"Initial Area: {A_i:.3e}")
print(f"Final Area: {A_f:.3e}")
print(f"Target Area: {A_0:.3e}\n")

# print triangle 
print("Triangles")
print(f"Initial Triangles: {jnp.mean(T_i):.3e}")
print(f"Final Triangles: {jnp.mean(T):.3e}")
print(f"Target Triangles: {T_0:.3e}\n")

print("Energy")
print(f"{helfrich_energy(res.params,hemesh,h_0,1e-2)/(8*jnp.pi*.01):.3e}")

# Plot results
plt.close('all')


Volume
Initial Volume: 4.153e+00
Final Volume: 3.051e+00
Target Volume: 2.932e+00

Area
Initial Area: 1.251e+01
Final Area: 1.229e+01
Target Area: 1.257e+01

Triangles
Initial Triangles: 9.770e-03
Final Triangles: 9.605e-03
Target Triangles: 9.817e-03

Energy
1.465e+00


In [ ]:
p = meshplot.plot(res.params, mesh.faces, shading={"wireframe": True}, return_plot=True)
p.add_mesh(geommesh.vertices + np.array([0, 0, 3]), mesh.faces, shading={"wireframe":True})


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0024898…

1

In [13]:
np.linalg.norm(mesh.vertices - res.params, axis=1).mean() / np.linalg.norm(mesh.vertices, axis=1).mean()


np.float64(0.481280656204326)